# GPU 虚拟化全景：MIG、MPS、Time-slicing 与 vGPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/lab-10-gpu-virtualization.ipynb)

把四种 GPU 共享/切分方式放进同一张图。本章以阅读和对照为主，用前九个 Lab 的直觉理解 MIG 硬隔离、MPS 软并发、time-slicing 时间片、vGPU 虚拟化各自的边界、代价与适用场景，看清 HAMi 的定位。

本 notebook 配合书中章节 [GPU 虚拟化全景：MIG、MPS、Time-slicing 与 vGPU](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应书中一个 Step；Markdown 单元摘录了必要的解释，完整上下文请回到书中阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## Why this problem matters

“GPU 共享/虚拟化”是个被滥用的词，背后其实是四种截然不同的机制。很多人把 HAMi、MIG、vGPU、MPS 混为一谈。本章把它们放进同一张比较图，让你在进入 Kubernetes 之前就建立清晰的边界，这样后面看到 HAMi 的 `gpumem`/`gpucores` 时，你会立刻知道它属于哪一类、替代/补充了谁。

## 四种机制全景

![一张物理 GPU 的四种切法：MIG 硬件切分、MPS 软件并发、Time-slicing 时间片轮转、vGPU 厂商分片](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/four-sharing-mechanisms.svg)

## 横向对照表

| 维度 | MIG | MPS | Time-slicing | vGPU |
| :--- | :--- | :--- | :--- | :--- |
| 切的是什么 | 物理的 SM / 显存 / L2 / 显存控制器分区 | 软件层的 kernel 并发 | 驱动按时间片轮流独占 | Hypervisor 层把一张卡虚拟成多个虚拟 GPU 给虚拟机 |
| 隔离强度 | **最强**（硬件级，故障与性能都隔离） | 弱（共享 server context，性能互相影响） | 进程级（context 独立，故障隔离；但性能和显存不隔离） | 强（每个 vGPU 显存固定，算力按调度策略分配） |
| 配额粒度 | 固定几档 profile（如 A100 的 1g.5gb … 7g.40gb） | 环境变量，启动时定 | 无；Kubernetes device plugin 只能按“副本数”超卖 | 厂商 profile（如 T4-4Q = 4 GB） |
| T4 是否支持 | ❌（需 Ampere+ 数据中心卡） | ✅ | ✅ | ✅ 硬件支持，但需要 NVIDIA vGPU 软件许可与 hypervisor，**Colab 里用不到** |
| 谁来做 | 驱动 + NVML（`nvidia-smi mig`） | 驱动自带的 MPS daemon | 驱动默认行为 | NVIDIA vGPU 软件栈 / 国产卡厂商栈 |
| HAMi 关系 | 可对接，提供硬隔离选项 | 概念原型，可选对接 | HAMi-core 在它之上加 `gpumem`/`gpucores` 配额 | 经由 device recipe 适配厂商栈 |

## 为什么 T4 支持不全反而是好教材

In [ ]:
!pip install -q nvidia-ml-py
from pynvml import *
nvmlInit()
h = nvmlDeviceGetHandleByIndex(0)
try:
    print("MIG:", nvmlDeviceGetMigMode(h))
except NVMLError_NotSupported:
    print("MIG on T4: not supported")
try:
    print("vGPU host mode:", nvmlDeviceGetVirtualizationMode(h))   # 0 = NONE，3 = HOST_VGPU，4 = HOST_VSGA
except NVMLError as e:
    print("virtualization mode:", e)
print("compute mode  :", nvmlDeviceGetComputeMode(h))   # 0 = Default，允许多进程

T4 报“不支持 MIG”，这正好说明：**不是所有 GPU 都能做硬隔离**。真实数据中心里，A100 / A30 / H100 / H200 / B200 这些卡才能 MIG，而大量在跑推理的 T4 / L4 / L40S 不能，这就逼出了 MPS 和 time-slicing 这类软件方案。**HAMi 存在的意义，很大程度上就是给那些“不能 MIG、但又要共享”的卡（以及 T4 这类入门卡）提供配额治理**。

`virtualization mode` 返回 `0`（NONE）则说明你拿到的是一张直通（passthrough）的整卡，而不是 vGPU 切出来的一片。Colab 的 GCE 实例就是这样：GPU 通过 PCIe passthrough 整张交给虚拟机，这是公有云 GPU 实例的主流做法。

## 三条决策线索

当你面对“这张卡要给多人用”时，按下面顺序判断：

1. **能不能 MIG？**（Ampere+ 数据中心卡且业务要强隔离）→ 用 MIG，最干净，代价是档位固定、切换 profile 要清空负载。
2. **能不能接受软共享、但要有配额？** → 这是 HAMi 的主战场（`gpumem`/`gpucores`），代价是性能隔离是软的、依赖 CUDA API 拦截。
3. **是虚拟机场景、或国产卡 / 需要厂商栈？** → 走 vGPU / device recipe（见 [L3 设备 Recipes](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../standards/vendor-device-recipes/)）。

![GPU 共享决策树：能 MIG 则用 MIG；否则要配额治理用 HAMi；否则厂商专用卡走 vGPU / device recipe](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/sharing-decision-tree.svg)

## Verification

完成后你应该能：

- 说出四种机制的切分对象与隔离强度；
- 解释 T4 为何不支持 MIG、这为什么反而重要；
- 解释“T4 硬件支持 vGPU”和“Colab 里用不到 vGPU”为什么不矛盾；
- 把 HAMi 定位到“time-slicing 之上的软共享 + 配额治理”这一格；
- 用决策图判断一个新场景该用哪种机制。

## Failure Modes

| 困惑 | 解答 |
| :--- | :--- |
| “HAMi 是不是就是 MIG？” | 不是。MIG 是硬件切分；HAMi 是 Kubernetes 之上的配额调度层，可对接 MIG 但不等于它。 |
| “vGPU 和 HAMi 冲突吗？” | 不冲突，vGPU 在 hypervisor 层，HAMi 在 Kubernetes 层；虚拟机里的 vGPU 对 HAMi 来说就是一张“小一点的卡”。国产卡走 device recipe，见 [L3](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/../standards/vendor-device-recipes/)。 |
| “MPS = HAMi 吗？” | 不是。[Lab 09](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/) 已说明 HAMi-core 走的是 CUDA API 拦截，不是 MPS。 |
| “Time-slicing 不是已经有隔离了吗？” | 它只有 context 级的故障隔离，没有显存上限和算力配额，这正是 HAMi 补的部分。 |

## 记录你的 Metadata Card

运行完成后，把实际数值回填到书中该章的 Metadata Card：运行日期、驱动 / CUDA 版本、关键输出数值、遇到的 Failure Mode。欢迎在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提交 PR 或 Issue。